## Cell 1: Check GPU Availability

In [1]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.9.0+cpu
CUDA available: False


## Cell 3: Clone Repository from GitHub

In [2]:
# Clone or update repository (safe for "Run All")
from pathlib import Path
import os


cwd = Path.cwd()
# If we're already inside the repo, use current directory.
if (cwd / 'config.yaml').exists() and (cwd / 'market_data').exists():
    repo_root = cwd
else:
    # Otherwise, assume repo lives at ./ml_engine (Colab default)
    repo_root = cwd / 'ml_engine'

if not repo_root.exists():
    subprocess.run(["git", "clone", "https://github.com/Raynergy-svg/ml_engine.git"], check=True)
    print("✓ Repository cloned")
else:
    if (repo_root / '.git').exists():
        print("✓ Repository exists, updating (fast-forward only)...")
        try:
            subprocess.run(["git", "fetch", "origin", "main"], cwd=str(repo_root), check=True)
            subprocess.run(["git", "checkout", "main"], cwd=str(repo_root), check=True)
            subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=str(repo_root), check=True)
            print("✓ Repository updated")
        except subprocess.CalledProcessError:
            print("⚠️ Could not auto-update repo (possibly local edits in the runtime).")
            print("Proceeding with the existing checkout.")
    else:
        raise RuntimeError(f"Found {repo_root} but it's not a git repo")

%cd {repo_root}
print(f"✓ Working directory: {os.getcwd()}")

NameError: name 'subprocess' is not defined

## Cell 6: Install Dependencies

In [ ]:
!pip install -q torch numpy pandas scikit-learn pyyaml tqdm rich matplotlib seaborn

## Cell 8: Update Config for GPU Training

In [ ]:
import yaml

# Load config
with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Update for GPU and faster training
config['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
config['batch_size'] = 128  # Larger batch size for GPU
config['epochs'] = 100  # Adjust as needed
config['auto_resume'] = True  # Enable auto-resume
config['mixed_precision'] = True  # Enable mixed precision for faster training
config['early_stopping_patience'] = 20

# Save updated config
with open('config.yaml', 'w') as f:
    yaml.dump(config, f)

print("Config updated:")
print(f"  Device: {config['device']}")
print(f"  Batch size: {config['batch_size']}")
print(f"  Epochs: {config['epochs']}")

## Cell 10: Train the Model (With Technical Indicators)

This cell will:
- Load all CSV files from market_data/
- Enable feature engineering (SMA, EMA, RSI, MACD, Bollinger Bands, etc.)
- Add 50+ technical indicators for better predictions
- Train the model with enhanced features

In [ ]:
from ml_engine_enhanced import EnhancedMLEngine
from data_loader import MarketDataLoader
from utils import load_config
import pandas as pd
import glob

# Load config
config = load_config('config.yaml')

# Load market data from all CSV files (exclude predictions files)
print("Loading market data...")
csv_files = glob.glob('market_data/*.csv')
csv_files = [f for f in csv_files if 'predictions' not in f]
print(f"Found {len(csv_files)} CSV files")

if not csv_files:
    raise ValueError("No CSV files found in market_data/ folder.")

# Load and combine all CSV files with proper datetime handling
dfs = []
for file in csv_files:
    print(f"Loading {file}...")
    df_part = pd.read_csv(file)

    # Standardize column names to lowercase
    df_part.columns = df_part.columns.str.lower()

    # Convert date/datetime column to datetime and set as index (UTC -> tz-naive)
    if 'date' in df_part.columns:
        df_part['date'] = pd.to_datetime(df_part['date'], utc=True)
        df_part.set_index('date', inplace=True)
    elif 'datetime' in df_part.columns:
        df_part['datetime'] = pd.to_datetime(df_part['datetime'], utc=True)
        df_part.set_index('datetime', inplace=True)

    if not isinstance(df_part.index, pd.DatetimeIndex):
        df_part.index = pd.to_datetime(df_part.index, utc=True)

    if df_part.index.tz is not None:
        df_part.index = df_part.index.tz_localize(None)

    if 'symbol' not in df_part.columns:
        symbol = file.split('/')[-1].replace('_data.csv', '').replace('.csv', '')
        df_part['symbol'] = symbol

    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=False)
print(f"Loaded {len(df)} data points from {df['symbol'].nunique()} tickers")
print(f"Date range: {df.index.min()} to {df.index.max()}")

print("\n" + "="*60)
print("PREPROCESSING + FEATURE ENGINEERING")
print("="*60)

data_loader = MarketDataLoader(config)

# Prefer enhanced features; fall back if something breaks
try:
    preprocess_out = data_loader.preprocess(df, add_features=True)
    print("✓ Feature engineering enabled")
except Exception as e:
    print(f"✗ Feature engineering failed: {e}")
    print("Falling back to basic features...")
    preprocess_out = data_loader.preprocess(df, add_features=False)

if not (isinstance(preprocess_out, tuple) and len(preprocess_out) == 6):
    raise ValueError(f"Unexpected preprocess return: {type(preprocess_out)}")

X_train, y_train, X_val, y_val, X_test, y_test = preprocess_out
print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Val:   X={X_val.shape}, y={y_val.shape}")
print(f"Test:  X={X_test.shape}, y={y_test.shape}")

# Update model config with correct input size
config.setdefault('model', {})
config['model']['input_size'] = X_train.shape[2]

print("\n" + "="*60)
print("TRAINING")
print("="*60)

engine = EnhancedMLEngine(config)
result_initial = engine.train(
    X_train, y_train,
    X_val, y_val,
    epochs=config['epochs']
 )
result = result_initial

print("\n" + "="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"Resumed: {result.get('resumed', False)}")
print(f"Total epochs: {result.get('total_epochs', 'N/A')}")
print(f"Best validation loss: {result['best_val_loss']:.6f}")
print(f"Final train loss: {result['train_losses'][-1]:.6f}")
print(f"Final val loss: {result['val_losses'][-1]:.6f}")

## Cell 11: Continue Training (Optional)

Runs extra training sessions to try to improve the current best checkpoint. (This runs before the download cell when you use "Run All".)

In [ ]:
import time

# Extra training loop (optional)
num_training_sessions = 5
epochs_per_session = 100

print("="*60)
print("CONTINUATION TRAINING")
print("="*60)
print(f"Sessions: {num_training_sessions}")
print(f"Epochs/session: {epochs_per_session}")
print(f"Total extra epochs: {num_training_sessions * epochs_per_session}")

result2 = None

for session in range(1, num_training_sessions + 1):
    print(f"\n{'='*60}")
    print(f"SESSION {session}/{num_training_sessions}")
    print(f"{'='*60}")

    start_time = time.time()
    result2 = engine.train(
        X_train, y_train,
        X_val, y_val,
        epochs=epochs_per_session,
    )
    elapsed = time.time() - start_time

    # Keep `result` pointing at the most recent training run
    result = result2

    print(f"\nSession {session} done in {elapsed/60:.1f} min")
    print(f"  Resumed: {result.get('resumed', False)}")
    print(f"  Total epochs: {result.get('total_epochs', 'N/A')}")
    print(f"  Best val loss: {result['best_val_loss']:.6f}")
    print(f"  Final train loss: {result['train_losses'][-1]:.6f}")
    print(f"  Final val loss: {result['val_losses'][-1]:.6f}")

    if result.get('stopped_early', False):
        print("\n✓ Early stopping triggered")
        break

result_last = result
print("\n✓ Continuation training complete")
print("Best model path: trained_data/models/best_model.pth")
print("If you downloaded earlier, re-run the Download cell to get the latest checkpoint.")

## Cell 15: Evaluate Model (Most Recent Run)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def _as_1d(a):
    a = np.asarray(a)
    return a.reshape(-1)

# Evaluate on validation + test sets
val_metrics = engine.evaluate(X_val, y_val)
test_metrics = engine.evaluate(X_test, y_test)

print("Validation Metrics:")
for key, value in val_metrics.items():
    print(f"  {key}: {value:.6f}")

print("\nTest Metrics:")
for key, value in test_metrics.items():
    print(f"  {key}: {value:.6f}")

loss_type = str(config.get("loss_type", "mse")).lower()
train_val_losses = result.get("val_losses", [])

# Predictions (most recent model state)
pred_val_1d = _as_1d(engine.predict(X_val))
y_val_1d = _as_1d(y_val)

print("\nPrediction stats (val):")
print(f"  y_val:   min={float(np.min(y_val_1d)):.6f} max={float(np.max(y_val_1d)):.6f} mean={float(np.mean(y_val_1d)):.6f} std={float(np.std(y_val_1d)):.6f}")
print(f"  pred:    min={float(np.min(pred_val_1d)):.6f} max={float(np.max(pred_val_1d)):.6f} mean={float(np.mean(pred_val_1d)):.6f} std={float(np.std(pred_val_1d)):.6f}")

# Recompute the configured loss on the full validation set (sanity check for scale)
err = pred_val_1d - y_val_1d
abs_err = np.abs(err)
recomputed_loss = None
if loss_type == "mse":
    recomputed_loss = float(np.mean(err ** 2))
elif loss_type == "mae":
    recomputed_loss = float(np.mean(abs_err))
elif loss_type in ("huber", "smooth_l1"):
    delta = float(config.get("huber_delta", 1.0)) if loss_type == "huber" else 1.0
    quad = 0.5 * (err ** 2)
    lin = delta * (abs_err - 0.5 * delta)
    recomputed_loss = float(np.mean(np.where(abs_err <= delta, quad, lin)))

if train_val_losses and recomputed_loss is not None:
    print(f"\nLoss scale check (loss_type={loss_type}):")
    print(f"  last val_loss (training): {train_val_losses[-1]:.6f}")
    print(f"  recomputed on y_val:       {recomputed_loss:.6f}")

# Visual sanity checks (these should change when the model improves)
# 1) Scatter uses a random sample across the *full* val set
seed = int(config.get("random_seed", 42))
rng = np.random.default_rng(seed)
m = min(2000, len(y_val_1d))
sample_idx = rng.choice(len(y_val_1d), size=m, replace=False)
y_s = y_val_1d[sample_idx]
p_s = pred_val_1d[sample_idx]

# 2) Series plot uses a window centered around the most "interesting" region (largest deviation from mean)
n = min(250, len(y_val_1d))
idx_peak = int(np.argmax(np.abs(y_val_1d - float(np.mean(y_val_1d)))))
start = max(0, min(len(y_val_1d) - n, idx_peak - n // 2))
end = start + n

lo = float(min(np.min(y_s), np.min(p_s)))
hi = float(max(np.max(y_s), np.max(p_s)))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_s, p_s, s=10, alpha=0.5)
plt.plot([lo, hi], [lo, hi], linestyle='--', linewidth=1)
plt.xlabel("Actual (y_val)")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted (val sample)")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(y_val_1d[start:end], label="Actual", linewidth=1)
plt.plot(pred_val_1d[start:end], label="Predicted", linewidth=1)
plt.xlabel("Sample index")
plt.ylabel("Value")
plt.title(f"Val window [{start}:{end}] actual vs predicted")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## Cell 17: Download Trained Model (Final Step)

This downloads a zip containing `trained_data/` and `config.yaml` after all training has completed.

In [ ]:
from pathlib import Path
import shutil
import time
import zipfile

# Colab download helper (safe fallback outside Colab)
try:
    from google.colab import files  # type: ignore
    _IN_COLAB = True
except Exception:
    files = None
    _IN_COLAB = False

# Bundle trained artifacts (runs last)
repo_root = Path.cwd()
trained_dir = repo_root / 'trained_data'
if not trained_dir.exists():
    raise FileNotFoundError(f"Missing {trained_dir} - run training first")

best_model_path = trained_dir / 'models' / 'best_model.pth'
if best_model_path.exists():
    stat = best_model_path.stat()
    print("Best model file:")
    print(f"  path (runtime): {best_model_path}")
    print(f"  path (relative): trained_data/models/best_model.pth")
    print(f"  size: {stat.st_size / (1024**2):.2f} MB")
    print(f"  modified: {time.ctime(stat.st_mtime)}")
else:
    print(f"Warning: best model file not found at {best_model_path}")

# Optional: print which training result is being exported
try:
    print("Training summary (most recent run):")
    print(f"  total_epochs: {result.get('total_epochs', 'N/A')}")
    print(f"  best_val_loss: {result.get('best_val_loss', float('nan')):.6f}")
except Exception:
    pass

export_dir = repo_root / 'export_trained_artifacts'
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

# Include trained_data + config.yaml
shutil.copy2(repo_root / 'config.yaml', export_dir / 'config.yaml')
shutil.copytree(trained_dir, export_dir / 'trained_data')

archive_base = repo_root / 'trained_artifacts'
zip_path = shutil.make_archive(str(archive_base), 'zip', export_dir)

zip_path = Path(zip_path)
print(f"\nZip created (runtime): {zip_path}")
print(f"Zip size: {zip_path.stat().st_size / (1024**2):.2f} MB")
print("\nNote: this path is inside the notebook runtime (e.g. Colab /content).")
print("The download will appear in your computer's Downloads folder (browser default).")

with zipfile.ZipFile(zip_path, 'r') as zf:
    names = zf.namelist()
    must_have = [
        'config.yaml',
        'trained_data/models/best_model.pth',
    ]
    print("\nZip content check:")
    for p in must_have:
        print(f"  {'✓' if p in names else '✗'} {p}")
    print("\nFirst 25 entries in zip:")
    for n in names[:25]:
        print(f"  {n}")

if _IN_COLAB:
    files.download(str(zip_path))
    print(f"\nTriggered browser download: {zip_path.name}")
else:
    print(f"\nNot running in Colab; zip is on disk at: {zip_path}")